# 📍 Redfin Distance Calculator
Upload your Redfin CSV and calculate the distance from each listing to your reference address.

## Step 1 — Install & Import

In [28]:
!pip install geopy pandas --quiet

import pandas as pd
import time
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import ssl
import certifi


print('✅ Ready')

✅ Ready


## Step 2 — ⚙️ Set Your Config

In [29]:
# ── Path to your downloaded Redfin CSV ──
CSV_PATH = "Desktop/Redfin Scraper"

# ── Your reference address (workplace, school, etc.) ──
REFERENCE_ADDRESS = "516 Ocean Rd, Santa Barbara, CA 93106"

# ── Output file name ──
OUTPUT_FILE = "redfin_with_distances_new.csv"

## Step 3 — Load CSV & Find Address Column

In [30]:
df = pd.read_csv("/Users/anugrhatamang/Desktop/Redfin Scraper/Redfin_listings_copy.csv")

print(f"✅ Loaded {len(df)} listings")
print(f"\nColumns found:\n{list(df.columns)}")
print(f"\nFirst few rows:")
display(df.head(3))

✅ Loaded 18 listings

Columns found:
['address', 'unit', 'bathrooms', 'bedrooms', 'description', 'price_from', 'distance_to_ucsb_miles']

First few rows:


,address,unit,bathrooms,bedrooms,description,price_from,distance_to_ucsb_miles
0,6679 Abrego Rd,Upstairs,1.0,1,Deal\nFURNISHED UNITS Lease with us and receiv...,"1,132",NaN
1,6679 Abrego Rd,Downstairs,1.0,1,Deal\nFURNISHED UNITS Lease with us and receiv...,"1,150",NaN
2,6741 Del Playa Dr,NaN,3.0,7,Unique features\nLandscaping,"19,000",NaN


## Step 4 — Set Address Column
After running Step 3, look at the column names printed above and paste the address column name below.

In [31]:
ADDRESS_COLUMN = "address"  # keep this as whatever your address column is called

# Set any columns that don't exist in your CSV to None
CITY_COLUMN = None
STATE_COLUMN = None
ZIP_COLUMN = None

In [32]:
print(REFERENCE_ADDRESS)

516 Ocean Rd, Santa Barbara, CA 93106


In [33]:
df.columns.tolist()

['address',
 'unit',
 'bathrooms',
 'bedrooms',
 'description',
 'price_from',
 'distance_to_ucsb_miles']

In [34]:
df= df.drop(columns=['distance_to_ucsb_miles'])

## Step 5 — Geocode Reference Address

In [35]:
geolocator = Nominatim(user_agent="redfin_distance_calc")

# Geocode reference address
ref_location = geolocator.geocode(REFERENCE_ADDRESS, timeout=10)

if ref_location:
    ref_coords = (ref_location.latitude, ref_location.longitude)
    print(f"Reference coordinates: {ref_coords}")
else:
    raise ValueError("Reference address could not be geocoded")

Reference coordinates: (34.409938, -119.8534378)


## Step 6 — Calculate Distances
⏳ This may take a few minutes depending on how many listings you have (rate limited to 1 per second).

In [ ]:
import requests
import time
from geopy.geocoders import Nominatim
from geopy.distance import geodesic

geolocator = Nominatim(user_agent="your_app")
ORS_API_KEY = " "  # <-- paste your key here
# API Key Obtained by openrouteservice.org

def get_walking_distance(full_address, ref_coords):
    """Geocode address and return walking distance in miles via OpenRouteService."""
    if not ref_coords:
        return "N/A"
    try:
        time.sleep(1)  # Nominatim rate limit
        loc = geolocator.geocode(full_address, timeout=10)
        if not loc:
            return "N/A"

        # ORS expects [longitude, latitude]
        origin = [ref_coords[1], ref_coords[0]]
        destination = [loc.longitude, loc.latitude]

        response = requests.post(
            "https://api.openrouteservice.org/v2/directions/foot-walking",
            headers={"Authorization": ORS_API_KEY, "Content-Type": "application/json"},
            json={"coordinates": [origin, destination]}
        )

        if response.status_code == 200:
            data = response.json()
            meters = data["routes"][0]["summary"]["distance"]
            miles = round(meters / 1609.34, 1)
            return miles
        else:
            return "N/A"

    except Exception as e:
        print(f"Error for {full_address}: {e}")
        return "N/A"

distances = []
total = len(df)
for i, row in df.iterrows():
    addr = row["address"]
    dist = get_walking_distance(addr, ref_coords)
    distances.append(dist)
    print(f"[{i+1}/{total}] {addr}  →  {dist} mi")

df["distance_to_ucsb"] = distances
print(f"\n✅ Done! Walking distances calculated for {total} listings.")

[1/18] 6679 Abrego Rd   →  0.9 mi
[2/18] 6679 Abrego Rd  →  0.9 mi
[3/18] 6741 Del Playa Dr  →  0.7 mi
[4/18] 6773 Del Playa Dr  →  0.8 mi
[5/18] 6564 Del Playa Dr  →  0.3 mi
[6/18] 6535 El Nido Ln  →  0.2 mi
[7/18] 6565 Sabado Tarde Rd  →  0.3 mi
[8/18] 6703 Del Playa Dr   →  0.6 mi
[9/18] 6703 Del Playa Dr   →  0.6 mi
[10/18] 6525 Del Playa Dr   →  0.2 mi
[11/18] 6509 Seville Rd   →  0.3 mi
[12/18] 6587 Cordoba Rd   →  0.6 mi
[13/18] 6761 Del Playa Dr   →  0.8 mi
[14/18] 6509 Seville Rd   →  0.3 mi
[15/18] 6754 Abrego Rd   →  1.0 mi
[16/18] 6754 Abrego Rd   →  1.0 mi
[17/18] 6754 Abrego Rd   →  1.0 mi
[18/18] 850 Camino Pescadero   →  0.6 mi

✅ Done! Walking distances calculated for 18 listings.


In [41]:
df.columns.tolist()

['address',
 'unit',
 'bathrooms',
 'bedrooms',
 'description',
 'price_from',
 'distance_to_ucsb']

In [42]:
df.head

<bound method NDFrame.head of                   address          unit  bathrooms  bedrooms  \
0         6679 Abrego Rd       Upstairs        1.0         1   
1          6679 Abrego Rd    Downstairs        1.0         1   
2       6741 Del Playa Dr           NaN        3.0         7   
3       6773 Del Playa Dr           NaN        3.0         4   
4       6564 Del Playa Dr           NaN        3.0         4   
5         6535 El Nido Ln           NaN        2.5         3   
6    6565 Sabado Tarde Rd           NaN        1.0         1   
7      6703 Del Playa Dr       Unit A-1        1.0         1   
8      6703 Del Playa Dr       Unit A-2        1.0         2   
9      6525 Del Playa Dr         Unit 2        2.0         3   
10       6509 Seville Rd        Unit 11        1.0         0   
11       6587 Cordoba Rd            NaN        1.0         1   
12     6761 Del Playa Dr       Unit 201        2.0         4   
13       6509 Seville Rd        Unit 11        1.0         0   
14        

## Step 7 — Preview & Save Results

In [57]:

import os

df_sorted['distance_to_ucsb'] = df_sorted['distance_to_ucsb'].round(2)
print(df_sorted.head())

path = '/Users/anugrhatamang/Desktop/Redfin Scraper/redfin_isla_vista_uncleaned.csv'
df_sorted.to_csv(path, index=False)
print(f"✅ Saved to {path}")

               address     unit  bathrooms  bedrooms  \
5      6535 El Nido Ln      NaN        2.5         3   
9   6525 Del Playa Dr    Unit 2        2.0         3   
13    6509 Seville Rd   Unit 11        1.0         0   
10    6509 Seville Rd   Unit 11        1.0         0   
4    6564 Del Playa Dr      NaN        3.0         4   

                                          description price_from  \
5                      In-unit\nGas Range\nDishwasher     14,000   
9                                                 NaN     11,000   
13  In-unit\nPatio\nRefrigerator\nCommunity\nExtra...      2,600   
10  In-unit\nPatio\nRefrigerator\nCommunity\nExtra...      2,600   
4                 Unique features\nLandscaping\nTrash     15,000   

    distance_to_ucsb  
5                0.2  
9                0.2  
13               0.3  
10               0.3  
4                0.3  
✅ Saved to /Users/anugrhatamang/Desktop/Redfin Scraper/redfin_isla_vista_uncleaned.csv
